# MobileNetV3-Small student: TFLite FP32 + INT8 target-test eval (standalone)

Training chara shudhu `.tflite` file diye INT8 accuracy ber korar notebook.

**Setup (Kaggle right panel → Add Input):**
1. PlantDoc dataset: `nirmalsankalana/plantdoc-dataset`
2. Apnar upload kora dataset, jekhane `mobilenet_v3_small_student_int8.tflite` ar `mobilenet_v3_small_student_fp32.tflite` ache

Accelerator: **None (CPU)** e cholbe. Tarpor *Run All*.

FP32 = **0.7595** ashle split notebook er sathe hubohu ek, tahole INT8 number valid.

In [ ]:
# Input check: dataset gulo thik moto add hoyeche kina
import os
for r, ds, fs in os.walk('/kaggle/input'):
    if r.count(os.sep) <= 4:
        print(r)
    for f in fs:
        if f.endswith('.tflite'):
            print('   ->', os.path.join(r, f))

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/mdfaishalahmed
   -> /kaggle/input/datasets/mdfaishalahmed/tflite/mobilenet_v3_small_student_fp32.tflite
   -> /kaggle/input/datasets/mdfaishalahmed/tflite/mobilenet_v3_small_student_int8.tflite
/kaggle/input/datasets/nirmalsankalana


In [ ]:
# ============================================================
# STANDALONE: TFLite FP32 + INT8 target-test accuracy (no retraining needed)
# ============================================================
# Only needs: the .tflite files + the PlantDoc dataset.
# Reproduces the notebook's exact target-test split (seed 42, stratified 70/15/15)
# and the exact eval preprocessing (Resize 256 -> Resize 224 -> ToTensor, raw [0,1]).
# FP32 should come out at 0.7595 -- if it does, the split is identical and the INT8 number is trustworthy.

import os, glob
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import tensorflow as tf

I0000 00:00:1794386670.578274      49 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794386688.878652      49 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
# ---------------- paths (change if needed) ----------------
# PlantDoc root: Kaggle input path notebook bhede alada hoy, tai auto-detect
_cands = ["/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset", "/kaggle/input/plantdoc-dataset"]
_cands += [r for r, ds, _ in os.walk("/kaggle/input") if "plantdoc" in os.path.basename(r).lower()]
TARGET_ALL = next(c for c in _cands if os.path.isdir(c))
print("PlantDoc root:", TARGET_ALL)
# Upload the models folder as a Kaggle dataset; this finds the files anywhere under /kaggle/input
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + glob.glob(f"./**/{pattern}", recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]
TFLITE_INT8 = find("mobilenet_v3_small_student_int8.tflite")
TFLITE_FP32 = find("mobilenet_v3_small_student_fp32.tflite")
print("INT8:", TFLITE_INT8, "\nFP32:", TFLITE_FP32)

SEED, IMG_SIZE = 42, 224
CLASSES = ["Tomato_Septoria_Leaf_Spot", "Potato_Early_Blight", "Corn_Leaf_Blight"]
TARGET_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato Septoria leaf spot"],
    "Potato_Early_Blight":       ["Potato leaf early blight"],
    "Corn_Leaf_Blight":          ["Corn leaf blight"],
}

PlantDoc root: /kaggle/input/datasets/nirmalsankalana/plantdoc-dataset
INT8: /kaggle/input/datasets/mdfaishalahmed/tflite/mobilenet_v3_small_student_int8.tflite 
FP32: /kaggle/input/datasets/mdfaishalahmed/tflite/mobilenet_v3_small_student_fp32.tflite


In [ ]:
# ---------------- same indexing as Part 1A ----------------
def _norm(s):
    return s.strip().lower().replace(" ", "").replace("_", "")

paths, labels = [], []
for idx, cls in enumerate(CLASSES):
    alias = {_norm(a) for a in TARGET_CLASS_DIRS[cls]}
    for d in [dp for dp, _, _ in os.walk(TARGET_ALL) if _norm(os.path.basename(dp)) in alias]:
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                paths.append(os.path.join(d, f)); labels.append(idx)
print(f"PlantDoc images: {len(paths)}")

PlantDoc images: 522


In [ ]:
# ---------------- same split as stratified_split_indices ----------------
idx_all = list(range(len(labels)))
idx_tv, idx_test = train_test_split(idx_all, test_size=0.15, stratify=labels, random_state=SEED)
test_paths  = [paths[i] for i in idx_test]
test_labels = [labels[i] for i in idx_test]
print(f"Target test: {len(test_paths)} images (notebook had 79)")

Target test: 79 images (notebook had 79)


In [ ]:
# ---------------- same eval preprocessing ----------------
# geo_eval Resize(256,256) -> to_student Resize(224,224) -> ToTensor -> (Normalize 0.5 then *0.5+0.5 = undo)
def load_raw01(p):
    img = Image.open(p).convert("RGB")
    img = img.resize((256, 256), Image.BILINEAR).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return (np.asarray(img, np.float32) / 255.0)[None]          # (1,224,224,3) NHWC, [0,1]

X = [load_raw01(p) for p in test_paths]

In [ ]:
# ---------------- TFLite eval WITHOUT XNNPACK (the actual fix) ----------------
def run_tflite(path, is_int8):
    it = tf.lite.Interpreter(
        model_path=path,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
    it.allocate_tensors()
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    sc = ind['quantization_parameters']['scales']
    zp = ind['quantization_parameters']['zero_points']
    P = []
    for a in X:
        if is_int8 and len(sc) and sc[0] != 0:
            a = np.clip(np.round(a / sc[0] + zp[0]), -128, 127).astype(ind['dtype'])
        it.set_tensor(ind['index'], a); it.invoke()
        P.append(int(np.argmax(it.get_tensor(outd['index'])[0])))
    return accuracy_score(test_labels, P), f1_score(test_labels, P, average='weighted')

PYTORCH_ACC = 0.7595
res = {}
for nm, pth, i8 in [("TFLite FP32", TFLITE_FP32, False), ("TFLite INT8", TFLITE_INT8, True)]:
    a, f = run_tflite(pth, i8)
    res[nm] = (a, f, os.path.getsize(pth) / 1024)
    print(f"{nm}: acc={a:.4f}  f1={f:.4f}  size={res[nm][2]:.1f} KB")

if abs(res["TFLite FP32"][0] - PYTORCH_ACC) > 1e-4:
    print("\nWARNING: FP32 != 0.7595 -> test split/preprocessing differs from the notebook run, "
          "INT8 number is not directly comparable.")
else:
    print("\nFP32 matches the notebook (0.7595) -> same test split, INT8 number is valid.")

drop = PYTORCH_ACC - res["TFLite INT8"][0]
print(f"INT8 accuracy drop: {drop:+.4f}   (MobileNetV2 had +0.0253, ResNet-50 +0.0000)")
print("  < 0.02 fine | 0.02-0.05 consider QAT | > 0.05 QAT or mixed precision needed")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


TFLite FP32: acc=0.7595  f1=0.7659  size=3642.0 KB


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


TFLite INT8: acc=0.6456  f1=0.6031  size=1180.9 KB

FP32 matches the notebook (0.7595) -> same test split, INT8 number is valid.
INT8 accuracy drop: +0.1139   (MobileNetV2 had +0.0253, ResNet-50 +0.0000)
  < 0.02 fine | 0.02-0.05 consider QAT | > 0.05 QAT or mixed precision needed
